Name: Shane D. Canabo <br>
Name: Athena S. Villarin <br>
Year & Section: BSCS 3A AI <br>
Date: 02/12/26

## Dataset
| doc | class |
| --- | --- |
| Free money now!!! | SPAM |
| Hi mom, how are you? | HAM |
| Lowest price for your meds | SPAM |
| Are we still on for dinner? | HAM |
| Win a free iPhone today | SPAM |
| Let's catch up tomorrow at the office | HAM |
| Meeting at 3 PM tomorrow | HAM |
| Get 50% off, limited time! | SPAM |
| Team meeting in the office | HAM |
| Click here for prizes! | SPAM |
| Can you send the report? | HAM |

In [3]:
import re
from collections import Counter, defaultdict

docs = [
    ("Free money now!!!", "SPAM"),
    ("Hi mom, how are you?", "HAM"),
    ("Lowest price for your meds", "SPAM"),
    ("Are we still on for dinner?", "HAM"),
    ("Win a free iPhone today", "SPAM"),
    ("Let's catch up tomorrow at the office", "HAM"),
    ("Meeting at 3 PM tomorrow", "HAM"),
    ("Get 50% off, limited time!", "SPAM"),
    ("Team meeting in the office", "HAM"),
    ("Click here for prizes!", "SPAM"),
    ("Can you send the report?", "HAM"),
]

def tokenize(text):
    text = text.lower()
    return re.findall(r"[a-z0-9']+", text)

def build_bow(data):
    bow = defaultdict(Counter)
    vocab = set()
    for text, label in data:
        tokens = tokenize(text)
        bow[label].update(tokens)
        vocab.update(tokens)
    return bow, sorted(vocab)

def table_from_rows(rows, title=None):
    if title:
        print(title)
    if not rows:
        return
    headers = list(rows[0].keys())
    widths = {
        h: max(len(h), max(len(str(r.get(h, ""))) for r in rows))
        for h in headers
    }
    header_line = " | ".join(h.ljust(widths[h]) for h in headers)
    sep_line = "-+-".join("-" * widths[h] for h in headers)
    print(header_line)
    print(sep_line)
    for r in rows:
        print(" | ".join(str(r.get(h, "")).ljust(widths[h]) for h in headers))

# a) Bag of Words (word frequency)
bow, vocab = build_bow(docs)
bow_rows = []
for label in sorted(bow.keys()):
    for token in vocab:
        bow_rows.append({"class": label, "token": token, "count": bow[label][token]})
table_from_rows(bow_rows, title="Bag of Words (word frequency)")

Bag of Words (word frequency)
class | token    | count
------+----------+------
HAM   | 3        | 1    
HAM   | 50       | 0    
HAM   | a        | 0    
HAM   | are      | 2    
HAM   | at       | 2    
HAM   | can      | 1    
HAM   | catch    | 1    
HAM   | click    | 0    
HAM   | dinner   | 1    
HAM   | for      | 1    
HAM   | free     | 0    
HAM   | get      | 0    
HAM   | here     | 0    
HAM   | hi       | 1    
HAM   | how      | 1    
HAM   | in       | 1    
HAM   | iphone   | 0    
HAM   | let's    | 1    
HAM   | limited  | 0    
HAM   | lowest   | 0    
HAM   | meds     | 0    
HAM   | meeting  | 2    
HAM   | mom      | 1    
HAM   | money    | 0    
HAM   | now      | 0    
HAM   | off      | 0    
HAM   | office   | 2    
HAM   | on       | 1    
HAM   | pm       | 1    
HAM   | price    | 0    
HAM   | prizes   | 0    
HAM   | report   | 1    
HAM   | send     | 1    
HAM   | still    | 1    
HAM   | team     | 1    
HAM   | the      | 3    
HAM   | time     | 0

In [4]:
# b) Class priors for HAM and SPAM
total_docs = len(docs)
class_counts = Counter(label for _, label in docs)
priors = {label: class_counts[label] / total_docs for label in class_counts}

prior_rows = [
    {"class": label, "prior": priors[label]}
    for label in sorted(priors.keys())
 ]
table_from_rows(prior_rows, title="Class Priors")

Class Priors
class | prior              
------+--------------------
HAM   | 0.5454545454545454 
SPAM  | 0.45454545454545453


In [5]:
# c) Likelihoods of tokens given class (Laplace smoothing)
alpha = 1.0
likelihoods = {}
for label in sorted(bow.keys()):
    total_tokens = sum(bow[label].values())
    denom = total_tokens + alpha * len(vocab)
    likelihoods[label] = {
        token: (bow[label][token] + alpha) / denom
        for token in vocab
    }

like_rows = []
for label in sorted(likelihoods.keys()):
    for token in vocab:
        like_rows.append({
            "class": label,
            "token": token,
            "likelihood": likelihoods[label][token],
        })
table_from_rows(like_rows, title="Token Likelihoods (Laplace-smoothed)")

Token Likelihoods (Laplace-smoothed)
class | token    | likelihood          
------+----------+---------------------
HAM   | 3        | 0.025974025974025976
HAM   | 50       | 0.012987012987012988
HAM   | a        | 0.012987012987012988
HAM   | are      | 0.03896103896103896 
HAM   | at       | 0.03896103896103896 
HAM   | can      | 0.025974025974025976
HAM   | catch    | 0.025974025974025976
HAM   | click    | 0.012987012987012988
HAM   | dinner   | 0.025974025974025976
HAM   | for      | 0.025974025974025976
HAM   | free     | 0.012987012987012988
HAM   | get      | 0.012987012987012988
HAM   | here     | 0.012987012987012988
HAM   | hi       | 0.025974025974025976
HAM   | how      | 0.025974025974025976
HAM   | in       | 0.025974025974025976
HAM   | iphone   | 0.012987012987012988
HAM   | let's    | 0.025974025974025976
HAM   | limited  | 0.012987012987012988
HAM   | lowest   | 0.012987012987012988
HAM   | meds     | 0.012987012987012988
HAM   | meeting  | 0.03896103896103896 
HAM

In [6]:
# d) Classify test sentences
import math

def classify(text, priors, likelihoods):
    tokens = tokenize(text)
    scores = {}
    for label in priors:
        score = math.log(priors[label])
        for token in tokens:
            if token in likelihoods[label]:
                score += math.log(likelihoods[label][token])
        scores[label] = score
    return max(scores, key=scores.get), scores

tests = [
    "Limited offer, click here!",
    "Meeting at 2 PM with the manager.",
]
test_rows = []
for text in tests:
    predicted, scores = classify(text, priors, likelihoods)
    test_rows.append({
        "text": text,
        "predicted": predicted,
        "score_ham": scores.get("HAM"),
        "score_spam": scores.get("SPAM"),
    })
table_from_rows(test_rows, title="Test Sentence Classification")

Test Sentence Classification
text                              | predicted | score_ham           | score_spam        
----------------------------------+-----------+---------------------+-------------------
Limited offer, click here!        | SPAM      | -13.637552069131367 | -11.27798004476371
Meeting at 2 PM with the manager. | HAM       | -13.704691371968996 | -17.54707632846997


In [ ]:
# scikit-learn Multinomial Naive Bayes
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

train_texts = [text for text, _ in docs]
train_labels = [label for _, label in docs]

vectorizer = CountVectorizer()
X_train = vectorizer.fit_transform(train_texts)

model = MultinomialNB()
model.fit(X_train, train_labels)

sk_tests = [
    "Limited offer, click here!",
    "Meeting at 2 PM with the manager.",
]
X_test = vectorizer.transform(sk_tests)
preds = model.predict(X_test)

sk_rows = []
for text, pred in zip(sk_tests, preds):
    sk_rows.append({"text": text, "predicted": pred})
table_from_rows(sk_rows, title="scikit-learn Predictions")